In [3]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

DATASET_DIR = '../dataset/deepfake'
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Data Augmentation to prevent overfitting on unseen images
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='training'
)

val_generator = datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='binary', subset='validation'
)

# Load MobileNetV2 Backbone
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Unfreeze the top 30 layers for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Low learning rate for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train for 10 epochs
print("Fine-tuning Deepfake Detector for external image generalizability...")
model.fit(train_generator, epochs=10, validation_data=val_generator)

# Overwrite saved model
os.makedirs('../models/deepfake', exist_ok=True)
model.save('../models/deepfake/deepfake_detector.h5')
print("✅ Retrained Deepfake Model Saved Successfully!")

Found 1633 images belonging to 2 classes.
Found 408 images belonging to 2 classes.
Fine-tuning Deepfake Detector for external image generalizability...
Epoch 1/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 47s 705ms/step - accuracy: 0.5573 - loss: 0.7405 - val_accuracy: 0.5025 - val_loss: 0.8560
Epoch 2/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 36s 691ms/step - accuracy: 0.6589 - loss: 0.6274 - val_accuracy: 0.5368 - val_loss: 0.7612
Epoch 3/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 35s 677ms/step - accuracy: 0.6865 - loss: 0.5839 - val_accuracy: 0.5760 - val_loss: 0.7367
Epoch 4/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 35s 668ms/step - accuracy: 0.7208 - loss: 0.5453 - val_accuracy: 0.5466 - val_loss: 0.7582
Epoch 5/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 35s 665ms/step - accuracy: 0.7563 - loss: 0.5070 - val_accuracy: 0.5441 - val_loss: 0.7996
Epoch 6/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 35s 669ms/step - accuracy: 0.7502 - loss: 0.4967 - val_accuracy: 0.5466 - val_loss: 0.8074
Epoch 7/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 35s 667ms/step - accuracy: 0.7728 - 

✅ Retrained Deepfake Model Saved Successfully!


In [4]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load saved model
deepfake_model = load_model('../models/deepfake/deepfake_detector.h5')

def predict_deepfake_image(img_path):
    img = image.load_img(img_path, target_size=(128, 128))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    prediction = deepfake_model.predict(img_array)[0][0]
    
    # Class mapping check
    fake_idx = train_generator.class_indices.get('training_fake', 0)
    fake_probability = prediction if fake_idx == 1 else (1 - prediction)
    
    status = "🚨 AI-GENERATED / DEEPFAKE IMAGE" if fake_probability > 0.5 else "✅ AUTHENTIC / REAL IMAGE"
    
    print("\n" + "="*50)
    print("   MODULE 4: DEEPFAKE IMAGE DETECTION RESULT")
    print("="*50)
    print(f"Image File       : {os.path.basename(img_path)}")
    print(f"Status           : {status}")
    print(f"Manipulation Risk: {fake_probability * 100:.2f}%")
    print("="*50)

# Test with first image in training_fake folder
fake_sample_dir = os.path.join(DATASET_DIR, 'training_fake')
sample_file = os.path.join(fake_sample_dir, os.listdir(fake_sample_dir)[0])
predict_deepfake_image(sample_file)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step

   MODULE 4: DEEPFAKE IMAGE DETECTION RESULT
Image File       : easy_100_1111.jpg
Status           : 🚨 AI-GENERATED / DEEPFAKE IMAGE
Manipulation Risk: 68.59%
